quality check

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/INvideos.csv")

In [2]:
print("Total rows:", len(df))

Total rows: 37352


In [3]:
missing_values = df.isnull().sum()

print(missing_values)

video_id                    0
trending_date               0
title                       0
channel_title               0
category_id                 0
publish_time                0
tags                        0
views                       0
likes                       0
dislikes                    0
comment_count               0
thumbnail_link              0
comments_disabled           0
ratings_disabled            0
video_error_or_removed      0
description               561
dtype: int64


In [4]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)

video_id                  0.000000
trending_date             0.000000
title                     0.000000
channel_title             0.000000
category_id               0.000000
publish_time              0.000000
tags                      0.000000
views                     0.000000
likes                     0.000000
dislikes                  0.000000
comment_count             0.000000
thumbnail_link            0.000000
comments_disabled         0.000000
ratings_disabled          0.000000
video_error_or_removed    0.000000
description               1.501928
dtype: float64


In [5]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 4263


In [6]:
unique_videos = df["video_id"].nunique()

print("Unique videos:", unique_videos)

Unique videos: 16307


In [7]:
numeric_columns = [
    "views",
    "likes",
    "dislikes",
    "comment_count"
]

for column in numeric_columns:
    negative_count = (df[column] < 0).sum()
    print(f"{column}: {negative_count} negative values")

views: 0 negative values
likes: 0 negative values
dislikes: 0 negative values
comment_count: 0 negative values


In [8]:
for column in numeric_columns:
    zero_count = (df[column] == 0).sum()
    print(f"{column}: {zero_count} zero values")

views: 0 zero values
likes: 781 zero values
dislikes: 790 zero values
comment_count: 1322 zero values


In [9]:
print(df["trending_date"].head())

0    17.14.11
1    17.14.11
2    17.14.11
3    17.14.11
4    17.14.11
Name: trending_date, dtype: str


In [10]:
print(df["publish_time"].head())

0    2017-11-12T12:20:39.000Z
1    2017-11-13T05:43:56.000Z
2    2017-11-12T15:48:08.000Z
3    2017-11-12T07:08:48.000Z
4    2017-11-13T01:14:16.000Z
Name: publish_time, dtype: str


In [16]:
# Convert trending date using the actual dataset format
trending_dates = pd.to_datetime(
    df["trending_date"],
    format="%y.%d.%m",
    errors="coerce"
)

# Convert publish timestamp
publish_dates = pd.to_datetime(
    df["publish_time"],
    errors="coerce",
    utc=True
)

# Check invalid dates
print(
    "Invalid trending dates:",
    trending_dates.isna().sum()
)

print(
    "Invalid publish dates:",
    publish_dates.isna().sum()
)

Invalid trending dates: 0
Invalid publish dates: 0


In [17]:
invalid_time_records = (
    publish_dates.dt.date >
    trending_dates.dt.date
)

print(
    "Videos published after trending date:",
    invalid_time_records.sum()
)

Videos published after trending date: 0


In [18]:
df.loc[
    invalid_time_records,
    ["video_id", "title", "publish_time", "trending_date"]
].head(20)

,video_id,title,publish_time,trending_date


In [19]:
print(df["video_error_or_removed"].value_counts())

video_error_or_removed
False    37341
True        11
Name: count, dtype: int64


In [20]:
error_percentage = (
    df["video_error_or_removed"].sum()
    / len(df)
) * 100

print(f"Error/removed percentage: {error_percentage:.2f}%")

Error/removed percentage: 0.03%


In [21]:
df = df[df["video_error_or_removed"] == False]

In [22]:
print(df["video_error_or_removed"].value_counts())

video_error_or_removed
False    37341
Name: count, dtype: int64


In [23]:
print(df["comments_disabled"].value_counts())

comments_disabled
False    36137
True      1204
Name: count, dtype: int64


In [24]:
comments_disabled_percentage = (
    df["comments_disabled"].sum()
    / len(df)
) * 100

print(
    f"Comments disabled: "
    f"{comments_disabled_percentage:.2f}%"
)

Comments disabled: 3.22%


In [25]:
disabled_zero_comments = df[
    (df["comments_disabled"] == True) &
    (df["comment_count"] == 0)
]

print(
    "Videos with comments disabled and zero comments:",
    len(disabled_zero_comments)
)

Videos with comments disabled and zero comments: 1204


In [26]:
print(df["ratings_disabled"].value_counts())

ratings_disabled
False    36560
True       781
Name: count, dtype: int64


In [27]:
ratings_disabled_percentage = (
    df["ratings_disabled"].sum()
    / len(df)
) * 100

print(
    f"Ratings disabled: "
    f"{ratings_disabled_percentage:.2f}%"
)

Ratings disabled: 2.09%


In [28]:
disabled_zero_likes = df[
    (df["ratings_disabled"] == True) &
    (df["likes"] == 0)
]

print(
    "Videos with ratings disabled and zero likes:",
    len(disabled_zero_likes)
)

Videos with ratings disabled and zero likes: 781


In [29]:
quality_report = {
    "Total Rows": len(df),
    "Total Columns": len(df.columns),
    "Duplicate Rows": df.duplicated().sum(),
    "Missing Cells": df.isnull().sum().sum(),
    "Invalid Trending Dates": trending_dates.isna().sum(),
    "Invalid Publish Dates": publish_dates.isna().sum(),
    "Negative Views": (df["views"] < 0).sum(),
    "Negative Likes": (df["likes"] < 0).sum(),
    "Negative Comments": (df["comment_count"] < 0).sum(),
    "Error/Removed Videos": df["video_error_or_removed"].sum(),
    "Comments Disabled": df["comments_disabled"].sum(),
    "Ratings Disabled": df["ratings_disabled"].sum()
}

quality_report

{'Total Rows': 37341,
 'Total Columns': 18,
 'Duplicate Rows': np.int64(4261),
 'Missing Cells': np.int64(561),
 'Invalid Trending Dates': np.int64(0),
 'Invalid Publish Dates': np.int64(0),
 'Negative Views': np.int64(0),
 'Negative Likes': np.int64(0),
 'Negative Comments': np.int64(0),
 'Error/Removed Videos': np.int64(0),
 'Comments Disabled': np.int64(1204),
 'Ratings Disabled': np.int64(781)}

In [30]:
quality_report_df = pd.DataFrame(
    list(quality_report.items()),
    columns=["Metric", "Value"]
)

quality_report_df

,Metric,Value
0,Total Rows,37341
1,Total Columns,18
2,Duplicate Rows,4261
3,Missing Cells,561
4,Invalid Trending Dates,0
5,Invalid Publish Dates,0
6,Negative Views,0
7,Negative Likes,0
8,Negative Comments,0
9,Error/Removed Videos,0


In [31]:
core_columns = [
    "video_id",
    "title",
    "channel_title",
    "trending_date",
    "publish_time",
    "views",
    "likes",
    "comment_count"
]

complete_rows = df[core_columns].notna().all(axis=1).sum()

total_rows = len(df)

quality_score = (complete_rows / total_rows) * 100

print(f"Data Quality Score: {quality_score:.2f}%")

Data Quality Score: 100.00%


In [32]:
if quality_score >= 95:
    status = "PASS"
elif quality_score >= 85:
    status = "WARNING"
else:
    status = "FAIL"

print("Data Quality Status:", status)

Data Quality Status: PASS


In [33]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("     TRENDPULSE DATA QUALITY")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

print(f"Total Rows:              {len(df):,}")
print(f"Duplicate Rows:          {df.duplicated().sum():,}")
print(f"Missing Cells:           {df.isnull().sum().sum():,}")
print(f"Invalid Trending Dates:  {trending_dates.isna().sum():,}")
print(f"Invalid Publish Dates:   {publish_dates.isna().sum():,}")
print(f"Negative Views:          {(df['views'] < 0).sum():,}")
print(f"Negative Likes:          {(df['likes'] < 0).sum():,}")
print(f"Negative Comments:       {(df['comment_count'] < 0).sum():,}")

print(f"Data Quality Score:      {quality_score:.2f}%")
print(f"Status:                  {status}")

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━
     TRENDPULSE DATA QUALITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total Rows:              37,341
Duplicate Rows:          4,261
Missing Cells:           561
Invalid Trending Dates:  0
Invalid Publish Dates:   0
Negative Views:          0
Negative Likes:          0
Negative Comments:       0
Data Quality Score:      100.00%
Status:                  PASS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
